# サンプリング波形の異常検知 設計議事録

## 0. 文書の目的

本議事録は、この会話で整理した **サンプリング波形の異常検知の設計方針** を、

- 検知したい背景
- 前提条件
- 候補手法と採否理由
- 特徴量設計
- 判定方法
- 可視化方法
- データ分割と評価方法
- 会話中に出た疑問点と、その回答

まで含めて、**漏れなく・構造化して** まとめたものである。

この文書は、実装前の設計メモ、レビュー用議事録、仕様書のたたき台として使うことを想定する。

---

## 1. 背景と最終ゴール

### 1.1 背景

ユーザーの目的は、**センサ波形の正常状態からの変化を異常として検知すること** である。

対象データは、1本のセンサについて、各観測点ごとに以下を持つ。

- センサ値
- 時間
- カム角度（0〜360度）

### 1.2 データの特徴

会話で確認された前提は以下である。

- 1周ごとに、**ほぼ同じ動作** が繰り返される
- ただし各周回の観測点は **不等間隔**
- 1周あたりの点数は **9〜12点程度** と少ない
- 正常データは **300周〜1000周程度** あり、ほとんどが正常
- 当面は **1本センサ** で考える
- 正常状態は、ひとまず **1種類で変わらない** と仮定する
- test には **異常データも入れたい**
- test の異常ラベルは **1周ごとに付いている**

### 1.3 最終ゴール

目標は、単に「異常/正常」を出すだけでなく、以下を両立することである。

- **いつ異常が出たか** を時系列で見えること
- **何が原因で異常になったか** がわかること
- 特に、**周波数成分が変わらず、振幅倍率だけが変わった場合も異常として検知できること**
- 高周波側の細かい変化についても、厳密な分類までは不要だが、**異常としては拾えること**

---

## 2. 最初に整理した基本方針

### 2.1 時間軸ではなく角度軸を主に使う

この会話では、データにカム角度があるため、

- 時間基準で波形を比較するより
- **角度基準で1周ごとに比較する**

方が自然だと整理した。

理由は、周期機械では回転速度のゆらぎがあると、時間軸では波形が横に伸び縮みして見えるからである。角度を基準にすると、**同じ角度で同じ機械状態を比べやすい**。

### 2.2 「1周ごと」を1サンプルとする

当初、「1周期ごとか、時間窓か」という話を整理した結果、本件では

- 2秒ごとのような時間窓よりも
- **0〜360度の1周を1サンプル**

として扱う方針を採用した。

理由は、

- 動作が1周ごとにほぼ繰り返される
- カム角度が取れている
- 異常ラベルも1周ごとに付けられる

ためである。

---

## 3. 当初候補に出た考え方と、その整理

### 3.1 生波形を細かく補間して直接比較する案

一時的に、各周回を細かい等間隔波形に補間して、その波形差で監視する案も考えられた。

しかし、会話の中で以下の理由から、第1候補にはしないと整理した。

- 1周あたり **9〜12点しか観測がない**
- そこから256点や360点へ細かく補間しても、増えた点の多くは「観測」ではなく「推定」
- そのため、**補間結果に依存しすぎる設計** になりやすい

結論として、**最初から密な波形差を主役にしない** 方が安全とした。

### 3.2 深い学習を最初から使う案

自己符号化器などの深い学習も候補として触れたが、第1候補にはしないと整理した。

理由は以下である。

- 1周ごとのデータ点数が少ない
- まずは **何が変わったか説明しやすい設計** がほしい
- 正常データが多いので、まずは正常の分布を素直に学ぶ方がよい

そのため、**第1版は手作り特徴量 + 軽い異常検知** で進める方針にした。

### 3.3 Isolation Forest を使う案

途中で「距離で判定するなら Isolation Forest でもよいのでは」という疑問が出た。

これに対して整理した内容は以下である。

- Isolation Forest は「距離法」ではなく、**ランダム分割でどれだけ早く孤立するか** を使う方法
- 今回は、
  - 正常が1種類と仮定できる
  - 各グループの特徴量が低次元
  - 正常分布の楕円境界や散布図を見せたい

  ため、**マハラノビス距離の方が自然**
- 一方で、IForest を比較候補に入れる価値はあるが、第1版の本命はマハラノビスとした

結論は、

- **第1版はマハラノビス距離で固定**
- IForest は比較候補として将来的に試す余地あり

である。

---

## 4. なぜ「少数の周期係数で表す」方針にしたか

### 4.1 採用した基本モデル

各周回を、角度 $\theta$ に対して、以下の低次の周期モデルで表す方針を採用した。

$$
y(\theta)
=
a_0
+
a_1\cos\theta+b_1\sin\theta
+
a_2\cos 2\theta+b_2\sin 2\theta
+
a_3\cos 3\theta+b_3\sin 3\theta
+\varepsilon(\theta)
$$

### 4.2 このモデルを選んだ理由

- 不等間隔データでも扱いやすい
- 1周9〜12点でも、3次程度までなら過度に自由度を増やしすぎない
- 1周の形を、
  - 全体の上下ずれ
  - 各次数の大きさ
  - 各次数の位相
  - 低次で説明できない残差

  に分解できる
- 後から「何が変わったか」を説明しやすい

### 4.3 なぜ 3 次までなのか

会話では、

- 1周9〜12点しかない
- 次数を上げすぎると不安定になる

ため、**第1版では 1〜3次まで** を採用した。

ここでの整理は、

- 低次のなめらかな周期成分はモデルで見る
- それより細かい成分は残差側に回す

という考え方である。

---

## 5. 「次数成分」という考え方の整理

会話の途中で、「振幅変化」「位相変化」「周波数成分変化」という言い方をしたため、**次数成分はどこに行ったのか** という疑問が出た。

その整理として、最終的に以下のように定義した。

### 5.1 次数成分そのもの

各次数の大きさと位相を

$$
A_k=\sqrt{a_k^2+b_k^2}
$$

$$
\phi_k=\mathrm{atan2}(b_k,a_k)
$$

とする。

ここで

- $A_1, A_2, A_3$ は **1次・2次・3次の振幅**
- $\phi_1, \phi_2, \phi_3$ は **1次・2次・3次の位相**

であり、これがそのまま **次数成分** である。

### 5.2 それをどう監視するか

次数成分は、そのまま以下の3面から監視すると整理した。

- **次数成分の振幅変化**
- **次数成分の位相変化**
- **次数成分の配分変化**

つまり、「次数成分」は消えたのではなく、**何を見るかで分けて言い直した** だけである。

---

## 6. 周波数成分の変化をどれで見るか

### 6.1 問題提起

ユーザーから、

- 「形状変化は、周波数成分の変化と振幅変化の両方を考慮できているのか」
- 「周波数成分の変化はどれで見るのか」

という疑問が出た。

### 6.2 会話の中での整理

最終的に、以下のように整理した。

#### 6.2.1 振幅倍率だけ変わる場合

もし波形が単純に $c$ 倍されるだけなら、

$$
a_k \to c a_k,\quad b_k \to c b_k
$$

となるため、

$$
A_k \to c A_k
$$

である。

したがって、**振幅倍率だけが変わる異常は、$A_1,A_2,A_3$ や RMS を見ていれば検知できる**。

この点はユーザーが特に重視していたため、設計上の重要ポイントとして明示した。

#### 6.2.2 低次の周波数成分変化

一方で、どの次数にどれだけ成分が載っているかの変化は、絶対振幅ではなく **配分** を見ることで捉える。

たとえば

$$
p_k=
\frac{A_k^2}{A_1^2+A_2^2+A_3^2}
$$

を使えば、1〜3次の中での成分配分の変化が見える。

したがって、

- **振幅倍率だけの変化** → 次数成分の振幅変化
- **低次の周波数成分変化** → 次数成分の配分変化

と分けた。

### 6.3 高周波数側の変化はどうなるか

ユーザーから、

- 「いまの設計でも高周波数でもわかるよね？ $p_k$ に出ないだけで」

という疑問が出た。

これに対して、以下のように整理した。

- **異常としては気づける可能性がある**
- ただし **高周波数成分の変化だと特定できるわけではない**

理由は、モデルに入れていない高次側の成分は、

$$
r(\theta)=y(\theta)-\hat y_{(1\sim3)}(\theta)
$$

という **残差** に落ちるからである。

そのため、

- 高周波成分の増加
- スパイク
- ノイズ増加
- サンプリングの偏りによる不安定さ

などは、いずれも残差側に現れうる。

### 6.4 最終的な割り切り

ユーザーは最後に、

- 「高周波成分だと特定できなくても、異常とわかればよい」

と整理した。

それを受けて、最終的には

- **低次の周波数成分変化** は配分変化スコアで見る
- **高次側や細かい変化** は「局所・高次残差異常」としてまとめて異常検知する

という方針で固定した。

---

## 7. 最終的に採用した監視の5本立て

会話の終盤で、最終的な監視段は以下の5本に整理された。

### 7.1 上下ずれ

目的:

- 全体が上にずれた / 下にずれた
- 全体強度が少し変わった

特徴量候補:

- $a_0$
- 全体RMS

### 7.2 次数成分の振幅変化

目的:

- 周波数成分の位置関係はそのままで、全体の振幅倍率だけが変わる異常を拾う
- 1次〜3次の絶対振幅の増減を拾う

特徴量候補:

- $A_1, A_2, A_3$
- 必要なら $A_1+A_2+A_3$
- 場合により RMS も参考にする

### 7.3 次数成分の位相変化

目的:

- 山や谷の位置ずれ、横ずれを拾う

特徴量候補:

- $\phi_1, \phi_2, \phi_3$ から作る位相差

位相差は線形の引き算ではなく、

$$
\delta_k=\mathrm{wrap}(\phi_k-\bar{\phi}_k)
$$

のように **巻き戻して使う** と整理した。

### 7.4 低次の周波数成分変化

目的:

- 1〜3次のどこに成分が多いか、その配分変化を拾う

特徴量候補:

- $p_1,p_2,p_3$
- または $A_2/(A_1+\varepsilon), A_3/(A_1+\varepsilon)$

### 7.5 局所・高次残差異常

目的:

- 3次までで説明できない細かい変化を拾う
- スパイク
- 高次側の変化
- 局所的な崩れ

特徴量候補:

- 残差RMS
- $P95(|r|)$
- $\max |r|$

---

## 8. 各特徴量が「なぜよいか」の整理

ここでは、会話の中で選んだ各特徴量の理由をまとめる。

### 8.1 $a_0$ と RMS

選んだ理由:

- 波形全体のレベルや強さの変化を単純に表せる
- 上下ずれや、全体的な振幅の変化に敏感
- 少ない点数でも比較的安定して計算できる

### 8.2 $A_1, A_2, A_3$

選んだ理由:

- 不等間隔でも低次の周期成分を要約できる
- 「振幅倍率だけが変わる」異常を直接拾える
- 1〜3次の強さそのものを見られる

### 8.3 $\phi_1, \phi_2, \phi_3$ から作る位相差

選んだ理由:

- 横ずれや山谷位置の変化を表せる
- 波形全体のタイミングずれに対応できる

注意点:

- 角度は 359度と1度が近いので、**そのまま線形変数として扱ってはいけない**
- 巻き戻しや sin/cos 変換など、円環量として扱う必要がある

### 8.4 配分指標 $p_k$ または比率

選んだ理由:

- 絶対振幅ではなく、どの次数に成分が多いかを見られる
- 振幅倍率だけが変わったときには不変に近く、**形の変化 / 周波数成分配分の変化** に効く

### 8.5 残差系指標

選んだ理由:

- 低次モデルで説明できない変化をまとめて拾える
- スパイクや細かい崩れに強い
- 高次側の変化があったときも、まず異常としては気づける

### 8.6 品質指標

追加で必要とした理由:

- 0度の観測点が毎回あるわけではない
- 角度が一部に偏ると、係数推定が不安定になる
- それを異常と混同しないために、判定の信頼性を別で持つ必要がある

品質指標として挙げたもの:

- 観測点数
- 角度カバー
- 最大角度ギャップ

---

## 9. なぜ個別PCAではなく、各段でマハラノビス距離にしたか

ユーザーから、

- 「4本に分けるが、それぞれの判定はどうするのか。個別にPCAか」

という疑問が出た。

それに対して整理した結論は以下である。

### 9.1 個別PCAを採用しなかった理由

- 各グループの特徴量は **2〜3次元程度** と小さい
- PCA は、より高次元で相関した特徴を圧縮する時に特に有効
- 2〜3次元なら、まずは **そのまま距離で判定** する方が自然
- 結果の解釈もしやすい

### 9.2 マハラノビス距離を採用した理由

- 正常が1種類と仮定できる
- 各グループの正常分布を「1つの雲」として扱いやすい
- 共分散を考慮した距離で見られる
- 散布図に **正常楕円** をそのまま描ける
- どの特徴が大きかったか説明しやすい

### 9.3 最終判断

- **各段は個別PCAではなく、小次元のロバスト・マハラノビス距離**
- PCA は今後、もし高次元化したら検討する余地あり

---

## 10. 総合スコアを最大値にした理由

ユーザーは、

- 「どれか1本で異常」

という方針を明確に示した。

これにより、総合スコアは和ではなく最大値とした。

### 10.1 定義

各段の正規化スコアを

$$
u_{\text{level}},
 u_{\text{amp}},
 u_{\text{phase}},
 u_{\text{freq}},
 u_{\text{resid}}
$$

とすると、総合は

$$
u_{\text{total}}
=
\max
\left(
 u_{\text{level}},
 u_{\text{amp}},
 u_{\text{phase}},
 u_{\text{freq}},
 u_{\text{resid}}
\right)
$$

である。

### 10.2 最大値を採用した理由

- 1本でも危険なら総合でも確実に上がる
- どの段が最大だったかで **主因がそのままわかる**
- 和にすると、軽いズレが複数重なった場合と、1本だけ大きい場合が混ざりやすい
- ユーザーの「どれか1本で異常」という設計意図にそのまま一致する

---

## 11. 0度の点が毎回ない問題への対応

ユーザーから、

- 「毎回ゼロがあるとはかぎらない」

という重要な情報が出た。

### 11.1 ここでの結論

- **0度の点が毎回なくても問題ない**
- 必要なのは「各観測点の角度がわかること」である

### 11.2 ただし注意点

- 角度が狭い範囲に偏ると、周期係数の推定が不安定になる
- そのため、異常判定とは別に **品質指標** を持つ必要がある

### 11.3 周回の切り出し

0度の観測点そのものがなくても、カム角度が 360→0 に折り返す箇所を使って、周回境界を推定する方針とした。

---

## 12. 可視化の設計

### 12.1 ユーザーの疑問

ユーザーは、

- 「距離で判定すると、散布図が主になるのでは」
- 「MSPCならスコアを時系列で出せるが、距離でも時系列化できるのか」

と疑問を出した。

### 12.2 回答として整理した内容

マハラノビス距離も、各周回に対して **1個の数** を返すので、

- 横軸: 周回番号または時刻
- 縦軸: スコア

で、そのまま **時系列グラフ化できる**。

したがって、距離法でも本番画面は散布図ではなく、**管理図のような折れ線** が主役でよい。

### 12.3 可視化の主画面

最終的に、以下の5段表示を採用した。

- 総合スコア
- 上下ずれ
- 次数成分の振幅変化
- 次数成分の位相変化
- 低次の周波数成分変化
- 局所・高次残差異常

※ 段数の表現は会話中に4本→5本へ拡張されたため、最終的には **総合1本 + 内訳5本** の考え方で整理するのが自然である。

### 12.4 散布図も欲しいという要望

ユーザーからさらに、

- 「散布図も欲しい」
- 「マハラビの正常な分布（等高線・楕円）とテストのデータ分布を見たい」

という要望が出た。

これを受けて、可視化は二本立てに整理した。

#### 12.4.1 本番監視用

- 時系列の折れ線グラフ

#### 12.4.2 開発・説明用

- 正常データの雲
- ロバスト共分散から作った正常楕円
- test データの点

を重ねた散布図

### 12.5 各散布図で何を描くか

- 上下ずれ: $(a_0, \mathrm{RMS})$
- 振幅変化: $(A_1,A_2)$, $(A_1,A_3)$, $(A_2,A_3)$
- 位相変化: $(\delta_1,\delta_2)$, $(\delta_1,\delta_3)$, $(\delta_2,\delta_3)$
- 低次の周波数成分変化: $(p_1,p_2)$, $(p_1,p_3)$, $(p_2,p_3)$
- 局所・高次残差異常: $(\mathrm{resid\_RMS}, \max|r|)$

---

## 13. データ分割の整理

### 13.1 「最後の確認用」とは何か

途中で、

- 「最後の確認用とは？」
- 「テストデータみたいな認識でいい？」

というやり取りがあった。

これに対して、

- 学習用 = モデルを作る
- しきい値決め用 = 境界を決める
- 最後の確認用 = どちらにも使っていない **純粋なテスト**

と整理した。

最終的に、ユーザーの認識としては

- **最後の確認用 = テストデータ**

で問題ないとした。

### 13.2 最終的に採用した分割の考え方

ユーザーは、

- 「学習データ（2つ）と test で分ける方がいいかな」
- 「test には異常も入れたい」

と整理した。

これを受けて、分割は次の3つにした。

- **fit 用**: 正常のみ
- **threshold 用**: 正常のみ
- **test 用**: 正常 + 異常

### 13.3 この分け方を採用した理由

- fit 用で、正常の中心と共分散を学ぶ
- threshold 用で、スコアのしきい値を正常だけから決める
- test 用で、
  - 正常に対する誤報率
  - 異常に対する検知率

  の両方を確認できる

### 13.4 時系列順に切ることの重要性

会話では、データは時系列順に並んでいる前提で、

- ランダム分割よりも
- **前から後ろへ時間順に切る**

方が自然だと整理した。

---

## 14. 評価設計の整理

### 14.1 test に異常ラベルがあることの意味

ユーザーから、

- 「test の異常ラベルはあり」
- 「1周ごと」

という情報が出た。

これにより、評価はまず **1周単位** で整理できるとした。

### 14.2 最低限見るべき評価

最終的に、以下を評価指標として挙げた。

#### 14.2.1 2値評価

- 正常 vs 異常の混同行列
- 再現率
- 適合率
- F1

#### 14.2.2 異常種類ごとの検知率

もともとユーザーは、

- 上下ずれ
- 位置ずれ
- 振幅変化
- 低次の周波数成分変化
- 局所・高次残差異常

のどれも重視したいと整理していたため、種類ごとの検知率を見る必要があるとした。

#### 14.2.3 スコアのしきい値に依存しない評価

異常検知では異常が少ないことが多いため、PR-AUC も有用と整理した。

### 14.3 原因分析の考え方

最終判定は 2 値でよいが、異常時には

- 最大だった内訳スコアを「主因」として記録する

という考え方を採用した。

これにより、

- 判定はシンプル
- 原因分析は多面的

という両立を図る。

---

## 15. 最終的に固定した「特徴量一覧表」

以下が、第1版として最終的に整理した特徴量表である。

| 監視段 | 使う特徴量 | 主に拾いたい変化 | 判定法 |
|---|---|---|---|
| 上下ずれ | $a_0$, 全体RMS | 全体の上下ずれ、全体強度の変化 | ロバスト・マハラノビス距離 |
| 次数成分の振幅変化 | $A_1, A_2, A_3$, 必要なら和 | 振幅倍率だけの変化、低次振幅の増減 | ロバスト・マハラノビス距離 |
| 次数成分の位相変化 | $\delta_1, \delta_2, \delta_3$ | 山谷位置のずれ、横ずれ | 位相差を使ったスコア |
| 低次の周波数成分変化 | $p_1,p_2,p_3$ または比率 | 1〜3次の配分変化 | ロバスト・マハラノビス距離 |
| 局所・高次残差異常 | 残差RMS, $P95(\|r\|)$, $\max\|r\|$ | スパイク、高次側の変化、局所異常 | ロバスト・マハラノビス距離 |
| 品質指標 | 観測点数, 角度カバー, 最大角度ギャップ | 判定信頼性の確認 | 別管理 |

---

## 16. 本会話で特に重要だった疑問点と回答まとめ

### 16.1 「4本に分けたが、それぞれの判定はどうするのか」

回答:

- 個別PCAではなく、小次元の距離判定
- 最後だけ総合スコアで統合

### 16.2 「0度の点が毎回ない」

回答:

- 問題ない
- 必要なのは各観測点の角度情報
- ただし角度カバーは品質指標として別管理

### 16.3 「距離法だと散布図が主になるのでは」

回答:

- 距離も 1周ごとに 1個のスコアになるので、時系列グラフにできる
- 散布図は説明用・開発用の補助

### 16.4 「Isolation Forest でもよくないか」

回答:

- 比較候補としてはよい
- ただし今回は、低次元・正常1種類・楕円可視化との相性から、マハラノビスを本命にした

### 16.5 「形状変化だけで、振幅変化と周波数成分変化の両方を見られるのか」

回答:

- ひとまとめでは不十分
- 振幅変化と配分変化を分けた方が明確

### 16.6 「次数成分はどこに行ったのか」

回答:

- 消えていない
- 振幅・位相・配分という3つの見方に分けて言い直しただけ

### 16.7 「周波数成分の変化はどれで見るのか」

回答:

- 低次については、次数成分の配分変化で見る
- 高次側は、まず残差異常として拾う

### 16.8 「いまの設計でも高周波数でもわかるよね？」

回答:

- 異常として気づける可能性はある
- ただし高周波成分変化だと特定できるわけではない
- 高次側は残差に落ちるので、まず異常として拾えばよい、という整理にした

### 16.9 「重要なのは、周波数が変化せずに振幅倍率だけが変化したときも検知できるか」

回答:

- できる
- ただし比率だけでは落ちるので、$A_1,A_2,A_3$ や RMS のような絶対振幅を別で持つ必要がある

---

## 17. 最終設計まとめ

本会話の時点での第1版設計は、以下である。

### 17.1 入力単位

- 1周ごとに1サンプル
- 不等間隔角度観測（1周9〜12点）

### 17.2 波形表現

- 角度基準の 3 次までの周期モデル

### 17.3 監視段

- 上下ずれ
- 次数成分の振幅変化
- 次数成分の位相変化
- 低次の周波数成分変化
- 局所・高次残差異常
- 品質指標（別管理）

### 17.4 判定法

- 各段: ロバスト・マハラノビス距離
- 総合: 正規化した各段スコアの最大値
- 方針: **どれか1本が閾値超えなら異常**

### 17.5 可視化

- 主画面: 総合 + 内訳の時系列グラフ
- 補助画面: 正常楕円 + test 点の散布図

### 17.6 データ分割

- fit: 正常のみ
- threshold: 正常のみ
- test: 正常 + 異常
- 基本は時間順分割

### 17.7 評価

- 正常/異常の2値評価
- 異常種類ごとの検知率
- 時系列上での異常位置確認
- 必要に応じて PR-AUC

---

## 18. 次の実務ステップ

この議論を受けて、次にやるべきことは以下である。

1. データ形式を固定する
   - 1周ごとの区切り方
   - 角度とセンサ値の保持形式

2. fit / threshold / test を時間順に分割する

3. 各周回に 3 次までの周期モデルを当てる

4. 上記の特徴量表に沿って特徴量を作る

5. 各監視段ごとにロバスト・マハラノビス距離を学習する

6. 正常 threshold データから、各段の閾値を決める

7. 総合スコアを最大値で作る

8. test で
   - 時系列グラフ
   - 散布図
   - 混同行列
   - 異常種類別検知率

   を確認する

---

## 19. 参考として会話内で触れた外部の考え方

この会話では、設計の考え方を支える背景として、主に以下の流れを参照した。

- 不等間隔データに対する Lomb–Scargle / 一般化 Lomb–Scargle
- 回転系の order analysis / order tracking
- ロバスト共分散とマハラノビス距離による外れ検知
- 多変量管理図の考え方
- 時系列データの train / validation / test 分割

※ 本議事録は会話内容の整理が主目的のため、ここでは文献の厳密な書誌情報は省略した。必要なら別途、引用文献一覧として整理する。

---

## 20. 一言まとめ

この会話で最終的に合意した核は、次の一文に尽きる。

**「1周9〜12点の不等間隔データなので、生波形を無理に細かく比較するより、1周を低次の周期成分と残差で表し、上下ずれ・振幅変化・位相変化・低次の周波数成分変化・残差異常を別々に監視し、どれか1本が危険なら異常とする」**

